# exp-03 Providers: same request, price and latency

**Question.** For an identical request, what does each model in `app/llm.py` cost, how long does one call take
(p50/p90/p95/p99), and how does it do on full turns?

**Dataset.** 100 of the 300 examples (`results/sample_ids.json`). Two paid parts, each behind its own switch:
`RUN_LATENCY` (one step-1 call per example per model) and `RUN_PASSRATE` (the full turn per example per model).
Groq models are paced (plan limit 8k tokens/min), so the parts take ~50 and ~110 minutes.

In [ ]:
# Setup: works whether the notebook's folder or the repo root is the working directory.
import json, sys
from pathlib import Path
HERE = Path.cwd()
if HERE.name != "exp03_providers":
    HERE = next(p for p in [Path.cwd() / "eng-experiments" / "exp03_providers", Path.cwd() / "exp03_providers"] if p.exists())
sys.path[:0] = [str(HERE), str(HERE.parent)]
import os; os.environ.setdefault("TRACE_PRINT", "0")   # no per-span printing during runs

import pandas as pd
from IPython.display import Markdown, display
import common
import bench
from app import llm
pd.set_option("display.max_colwidth", 120)

## Settings

In [ ]:
RUN_LATENCY = False       # True = one step-1 call per example per model (costs money)
RUN_PASSRATE = False      # True = one full agent turn per example per model (costs money)
FIRST = None              # e.g. 2 for a smoke run; None = all 100
MODELS = [m["id"] for m in llm.available()]       # every model whose API key is set
examples = bench.sample()[:FIRST] if FIRST else bench.sample()
print(len(examples), "examples;", MODELS)
pd.Series([t for e in examples for t in e["tags"]]).value_counts().rename("examples per tag")

In [ ]:
# The identical request every model gets for one example (the last input item; the system prompt and tools are shared)
print(bench.request_for(examples[0])[-1]["content"])

In [ ]:
for part, table in (("latency", bench.USD_CALL), ("passrate", bench.USD_TURN)):
    usd = sum(table[m] for m in MODELS) * len(examples)
    print(f"{part}: {len(examples) * len(MODELS)} {'calls' if part == 'latency' else 'turns'}, about ${usd:.2f}")

## Part 1: one step-1 call per example per model (paid)

In [ ]:
if RUN_LATENCY:
    print("saved", bench.run_latency(examples, MODELS))
else:
    print("RUN_LATENCY is False: nothing was called.")

## Part 2: full turn per example per model (paid)

In [ ]:
if RUN_PASSRATE:
    print("saved", bench.run_passrate(examples, MODELS))
else:
    print("RUN_PASSRATE is False: nothing was called.")

## Results (all files in `results/`)

Also rewrites the tables in `README.md`.

In [ ]:
lat = [json.loads(l) for f in sorted((HERE / "results").glob("latency-*.jsonl")) for l in f.read_text().splitlines()]
pas = [json.loads(l) for f in sorted((HERE / "results").glob("passrate-*.jsonl")) for l in f.read_text().splitlines()]
display(Markdown(bench.report()))

In [ ]:
# Latency per model from the raw calls (ok calls only), ms
if lat:
    d = pd.DataFrame(lat)
    ok = d[d.status == "ok"]
    display(ok.groupby("model")["ms"].describe(percentiles=[.5, .9, .95, .99]))
    display(d.groupby(["model", "status"]).size().unstack(fill_value=0))